Data Ingestion


In [2]:
from langchain_core.documents import Document

In [3]:
doc = Document(
    page_content="This is a test document.",
    metadata={"source": "test_source.txt",
              "pages" : 1,
              "author" : "Rajiv",
              "date" : "2024-06-01"})
doc

Document(metadata={'source': 'test_source.txt', 'pages': 1, 'author': 'Rajiv', 'date': '2024-06-01'}, page_content='This is a test document.')

In [4]:
import os
os.makedirs("../data/text_files", exist_ok=True)

In [5]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [6]:
### TextLoader
from langchain_community.document_loaders import TextLoader

from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [7]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\np

In [8]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': '', 'creator': '', 'creationdate': '2024-03-04T11:16:13+00:00', 'source': '../data/pdf/AI-for-Education-RAG.pdf', 'file_path': '../data/pdf/AI-for-Education-RAG.pdf', 'total_pages': 18, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-03-04T11:16:13+00:00', 'trapped': '', 'modDate': "D:20240304111613+00'00'", 'creationDate': "D:20240304111613+00'00'", 'page': 0}, page_content='Using your own content in LLM’s -\nRetrieval Augmented Generation\n(RAG)'),
 Document(metadata={'producer': '', 'creator': '', 'creationdate': '2024-03-04T11:16:13+00:00', 'source': '../data/pdf/AI-for-Education-RAG.pdf', 'file_path': '../data/pdf/AI-for-Education-RAG.pdf', 'total_pages': 18, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-03-04T11:16:13+00:00', 'trapped': '', 'modDate': "D:20240304111613+00'00'", 'creationDate': "D:20240304111613+00'00'", 'page': 1}, page_content='Gen-AI

In [9]:
type(pdf_documents[0])

langchain_core.documents.base.Document

embedding And vectorstoreDB


In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
class EmbeddingStore:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None  
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"✅ Loaded model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"✅ Model loaded successfully: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            raise e
        

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model not loaded.")
        try:
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"✅ Generated embeddings for {len(texts)} texts.")
            return embeddings
        except Exception as e:
            print(f"❌ Error generating embeddings: {e}")
            raise e
        

embedding_store = EmbeddingStore()
embedding_store

✅ Loaded model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10851.88it/s]


✅ Model loaded successfully: 384


/tmp/ipykernel_3783/2163658623.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Model loaded successfully: {self.model.get_sentence_embedding_dimension()}")


VectorStore

In [ ]:
class vectorStore:
    def __init__(self, collection_name: str = "pdf_documents" , persist_directory: str = "../data/chroma_db"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()
    
    def _initialize_chromadb(self):
        try:
            self.client = chromadb.Client(Settings(chroma_db_impl="duckdb+parquet", persist_directory=persist_directory))
            if self.collection_name in self.client.list_collections():
                self.collection = self.client.get_collection(name=self.collection_name)
                print(f"✅ Loaded existing collection: {self.collection_name}")
            else:
                self.collection = self.client.create_collection(name=self.collection_name)
                print(f"✅ Created new collection: {self.collection_name}")
        except Exception as e:
            print(f"❌ Error initializing ChromaDB: {e}")
            raise e